In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal

from utils import find_distributions, \
find_api_endpoints, get_rowstore_data, get_metadata_field, NAMESPACE

import plotly.express as px
from IPython.display import HTML
import plotly
import pandas as pd

### Documentation

[EntryStore Documentation](https://entrystore.org)

[EntryStore API Documentation](https://entrystore.org/api)

### Examples of GET:

`{base}{context-id}/metadata/{entry-id}`

Catalog Metadata: https://zg-demo.entryscape.net/store/1/metadata/1

Dataset Metadata: https://zg-demo.entryscape.net/store/1/metadata/5

Distribution metadata: https://zg-demo.entryscape.net/store/1/metadata/7

# Get Metadata

Define IDs and catalog URL.

In [ ]:
context_id = "1"
dataset_id = "5"
distribution_id = "28"

catalog_url = f"https://zg-demo.entryscape.net/store/{context_id}"

Using rdflib, parse the rdf-graph of a dataset and print it:

In [ ]:
g = Graph()

# Parse dataset RDF
dataset_url = f"{catalog_url}/resource/{dataset_id}"

g.parse(dataset_url)

g.print(format="turtle")

Get the value of a single metadata field, e.g., `dcterms:title`

In [ ]:
get_metadata_field(
    "dcterms:title",
    catalog_url,
    dataset_id
)

Extract other metadata, for example the description the modification date.
Have a look at the metadata: https://zg-demo.entryscape.net/store/1/resource/5

In [ ]:
# get_resource_metadata_field(...)

Get the name of the Publisher

1. get the `publisher` URI
2. get the `name` of the publisher

In [ ]:
# publisher_uri = get_metadata_field(...)

# instead of the dataset_id, we use resource_uri=publisher_uri to get the publisher name
# publisher_name = get_metadata_field(...)


Find resources with format `application/json`

In [ ]:
find_distributions(
    catalog_url,
    dataset_id,
    format_mime="application/json"
)

Find resources with api endpoint

In [ ]:
api_endpoints = find_api_endpoints(
    catalog_url,
    dataset_id
)

# Get data from RowStore API

Query parameter can be regex

In [ ]:
query_params = {
    "datum/zeit": "^(2022|2023|2024).*12:00:00$" # extract one data point per day at noon
    }


In [ ]:
df_combined = pd.DataFrame()

for api_endpoint in api_endpoints:
    accessURL = api_endpoint.get("accessURL")
    title = api_endpoint.get("title")
    print(f"Fetching data from API for station: {title}")
    gw_data = get_rowstore_data(
        accessURL,
        query_params,
        fetch_all=True
    )

    df = pd.DataFrame(gw_data)
    df['gw-stand [m ü.m.]'] = pd.to_numeric(df['gw-stand [m ü.m.]'], errors='coerce')
    df = df.assign(station=title)
    df_combined = pd.concat([df_combined, df], ignore_index=True)

In [ ]:
# add a column with the difference to the mean gw-stand per station
df_combined['gw-stand-diff'] = df_combined['gw-stand [m ü.m.]'] - df_combined.groupby('station')['gw-stand [m ü.m.]'].transform('mean')
df_combined

In [ ]:
template = "plotly_dark"

fig = px.line(
    df_combined,
    x="datum/zeit",
    y="gw-stand-diff",
    color="station",
    labels={"datum/zeit": "Datum", "gw-stand-diff": "Grundwasserstand Differenz zum Mittelwert (m)"},
    template=template
    )

# legend title
fig.update_layout(legend_title_text='Messstation')
# addd caption with data source
fig.add_annotation(
    text=(f"Datenquelle: {publisher_name}" if 'publisher_name' in locals() else "Datenquelle: Opendata Zug"),
    xref="paper", yref="paper",
    x=0, y=-0.2,
    showarrow=False,
    font=dict(size=12, color="lightgrey")
)


fig.show()